In [69]:
import lancedb

db = lancedb.connect(uri="vector_database")
db

LanceDBConnection(uri='c:\\Users\\edwin\\Documents\\AI_Engineering_Course_Material\\AI_Engineering_Edwin_Lindblom\\code_alongs\\13_lancedb\\vector_database')

In [70]:
db.uri

'c:\\Users\\edwin\\Documents\\AI_Engineering_Course_Material\\AI_Engineering_Edwin_Lindblom\\code_alongs\\13_lancedb\\vector_database'

## Read in data

In [71]:
import json
with open ("data/animals_text_embeddings.json", "r") as file:
    data = json.loads(file.read())

data        

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

## Create table

In [72]:
db.create_table("animals", exist_ok=True, data=data)


LanceTable(name='animals', version=1, _conn=LanceDBConnection(uri='c:\\Users\\edwin\\Documents\\AI_Engineering_Course_Material\\AI_Engineering_Edwin_Lindblom\\code_alongs\\13_lancedb\\vector_database'))

In [73]:
db.list_tables()

ListTablesResponse(tables=['animals'], page_token=None)

In [74]:
db["animals"]

LanceTable(name='animals', version=1, _conn=LanceDBConnection(uri='c:\\Users\\edwin\\Documents\\AI_Engineering_Course_Material\\AI_Engineering_Edwin_Lindblom\\code_alongs\\13_lancedb\\vector_database'))

In [75]:
db["animals"].head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

In [76]:
df_animals =db["animals"].to_pandas()
df_animals

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"


In [77]:
df_animals.iloc[2]["text"], df_animals.iloc[2]["vector"]

('A large gray elephant drinking water.',
 array([0.88, 0.22, 0.55], dtype=float32))

to add more data

In [78]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]


db["animals"].add(more_data)

AddResult(version=2)

In [79]:
db["animals"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Create empty table
- create an empty table first and then place in the data

In [80]:
from lancedb.pydantic import LanceModel


class EmployeeSchema(LanceModel):
    first_name: str
    last_name: str
    salary: int

db.create_table(name= "employees", schema = EmployeeSchema)

LanceTable(name='employees', version=1, _conn=LanceDBConnection(uri='c:\\Users\\edwin\\Documents\\AI_Engineering_Course_Material\\AI_Engineering_Edwin_Lindblom\\code_alongs\\13_lancedb\\vector_database'))

In [81]:
data = [{"first_name": "Bibbi", "last_name": "Babblarna", "salary": 1000}]
db["employees"].add(data)

AddResult(version=2)

In [82]:
db["employees"].to_pandas()

,first_name,last_name,salary
0,Bibbi,Babblarna,1000


In [83]:
db.list_tables()

ListTablesResponse(tables=['animals', 'employees'], page_token=None)

In [84]:
db.drop_table("employees")

In [85]:
db.list_tables()

ListTablesResponse(tables=['animals'], page_token=None)

## Vector search

- ANN - approximate nearest neighbour for vector search

1. send in a query vector directly and search
    - this require that we embed our quert first using the same embedding as what was used in the knowledge base
2. send in a text and let lancdb automatically embed it and search


In [86]:
db["animals"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [87]:
# assume that we embed our question using the same embedding model as the one for animals
query_vector = [0.9, 0.2, 0.5]

db["animals"].search(query_vector).limit(4).to_pandas()

,text,vector,_distance
0,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]",0.0033
1,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]",0.0094
2,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.0573
3,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.2834


## Embeddings API

- let lancedb embed our documents automatically
- let lancedb embed our query automatically and search using natural language

In [96]:
from lancedb.pydantic import Vector
from lancedb.embeddings import get_registry

model = get_registry().get("gemini-text").create(name="gemini-embedding-001")

model

GeminiText(max_retries=7, name='gemini-embedding-001', query_task_type='retrieval_query', source_task_type='retrieval_document')

In [97]:
from dotenv import load_dotenv
load_dotenv()
embeddings = model.generate_embeddings("why are SQL good at relationships? Because they are relational")


In [100]:
import numpy as np
np.array(embeddings).shape

(62, 3072)

In [101]:
class JokeModel(LanceModel):
    joke: str = model.SourceField() # input till embedding function. this is the text we want to embed
    embedding: Vector(3072) = model.VectorField() # computed embedding in this column. the dimension must match the embedding model output dimension. this is where the vector will be stored

db.create_table(name="jokes", schema= JokeModel, exist_ok=True)


LanceTable(name='jokes', version=1, _conn=LanceDBConnection(uri='c:\\Users\\edwin\\Documents\\AI_Engineering_Course_Material\\AI_Engineering_Edwin_Lindblom\\code_alongs\\13_lancedb\\vector_database'))